In [1]:
"""
ecb_transcript_v2.py
====================
Parser de ruedas de prensa del BCE, 2013-2024.
 
Cambio de diseño respecto a la versión anterior:
  el troceado del Q&A se hace sobre la ESTRUCTURA HTML, no sobre texto plano.
 
  - Pregunta  = parrafo mayoritariamente en cursiva (<em>/<i>)
  - Respuesta = parrafo en texto plano precedido de una etiqueta de hablante
                en negrita (<strong>/<b>), p.ej. "Draghi:" o "Lagarde:"
  - Una respuesta puede abarcar VARIOS parrafos; solo el primero lleva etiqueta.
 
Unidad de observacion: UNA FILA = UNA RESPUESTA del BCE (como Gorodnichenko,
Pham & Talavera 2023: item = ..._A1, _A2, ...). La pregunta se guarda en columna
aparte para validar el emparejamiento, pero NO se puntua.
 
Salida: ECB_text_2013_2024_v2.xlsx
  remarks       : 1 fila por conferencia  [date, era, text]
  remarks_para  : 1 fila por parrafo      [date, order, text]
  answers       : 1 fila por respuesta    [date, item, speaker, question, text]
 
El statement tecnico separado (PDF, 2022+) se trata en un script aparte; no
entra en Text_m porque no es comparable en el tiempo.
 
Uso:
    python ecb_transcript_v2.py --diagnose      # inspecciona 3 ficheros
    python ecb_transcript_v2.py --speakers      # lista etiquetas encontradas
    python ecb_transcript_v2.py --run           # parsea todo + valida
"""

'\necb_transcript_v2.py\n====================\nParser de ruedas de prensa del BCE, 2013-2024.\n\nCambio de diseño respecto a la versión anterior:\n  el troceado del Q&A se hace sobre la ESTRUCTURA HTML, no sobre texto plano.\n\n  - Pregunta  = parrafo mayoritariamente en cursiva (<em>/<i>)\n  - Respuesta = parrafo en texto plano precedido de una etiqueta de hablante\n                en negrita (<strong>/<b>), p.ej. "Draghi:" o "Lagarde:"\n  - Una respuesta puede abarcar VARIOS parrafos; solo el primero lleva etiqueta.\n\nUnidad de observacion: UNA FILA = UNA RESPUESTA del BCE (como Gorodnichenko,\nPham & Talavera 2023: item = ..._A1, _A2, ...). La pregunta se guarda en columna\naparte para validar el emparejamiento, pero NO se puntua.\n\nSalida: ECB_text_2013_2024_v2.xlsx\n  remarks       : 1 fila por conferencia  [date, era, text]\n  remarks_para  : 1 fila por parrafo      [date, order, text]\n  answers       : 1 fila por respuesta    [date, item, speaker, question, text]\n\nEl statem

In [2]:
import argparse
import re
import sys
import time
from pathlib import Path
 
import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag

In [3]:
# ---------------------------------------------------------------- rutas
BASE = Path(r"C:\Users\PC\Desktop\TFM\BIG DATA\TEXT SENTIMENT\data")
URL_MASTER = BASE / "ecb_urls_master_2013_2024.csv"
HTML_DIR = BASE / "html_raw_2013_2024"
OUTPUT = BASE / "ECB_TEXT (2013-2024)_v2.xlsx"
 
HEADERS = {"User-Agent": "Mozilla/5.0 (academic research; TFM ECB communication)"}

In [4]:
# ---------------------------------------------------------------- hablantes
# Miembros del Comite Ejecutivo / presidentes cuyas respuestas SI cuentan.
# Curada a partir de --speakers; ampliala si aparece alguien nuevo.
ECB_POLICYMAKERS = {
    "draghi", "lagarde", "trichet",
    "constancio", "constâncio",
    "de guindos", "guindos",
    "praet", "coeure", "cœuré", "coeuré",
    "mersch", "lautenschlager", "lautenschläger",
    "lane", "schnabel", "elderson", "panetta", "cipollone", "buch",
}
# Etiquetas que NO son politica monetaria (moderadores, prensa, protocolo).
NON_POLICYMAKERS = {"graeff", "question", "questions", "moderator"}
 
# Anclas que marcan el final de la declaracion introductoria.
QA_ANCHORS = [
    r"we are now at your disposal for questions",
    r"we are now ready to take your questions",
    r"we are now (?:happy|pleased) to take your questions",
    r"transcript of the questions and answers",
    r"questions\s+and\s+answers",
]
 
MIN_PARA_CHARS = 50   # filtro de parrafos basura en remarks_para

In [5]:
 
# ---------------------------------------------------------------- utilidades
def clean_url(u: str) -> str:
    """El master tiene alguna URL con doble barra: .eu//press/..."""
    return re.sub(r"(?<!:)//+", "/", u.strip())
 
 
def local_path(date: str) -> Path:
    return HTML_DIR / f"ecb_{date.replace('-', '')}.html"
 
 
def download_all(force: bool = False, pause: float = 1.0) -> None:
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    urls = pd.read_csv(URL_MASTER)
    for _, row in urls.iterrows():
        dest = local_path(row["target_date"])
        if dest.exists() and not force:
            continue
        url = clean_url(row["url"])
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            dest.write_text(r.text, encoding="utf-8")
            print(f"  ok  {row['target_date']}")
        except Exception as exc:
            print(f"  FALLO {row['target_date']}: {exc}", file=sys.stderr)
        time.sleep(pause)
 
 
def get_soup(date: str) -> BeautifulSoup:
    return BeautifulSoup(local_path(date).read_text(encoding="utf-8"), "html.parser")
 
 
def article_paragraphs(soup: BeautifulSoup) -> list:
    """Todos los <p> del cuerpo del articulo, sin navegacion ni pie."""
    main = (soup.find("main")
            or soup.find("div", class_="section")
            or soup.find(id="main-content")
            or soup)
    return [p for p in main.find_all("p") if p.get_text(strip=True)]

In [6]:
# ---------------------------------------------------------------- clasificacion
def italic_ratio(p: Tag) -> float:
    """Fraccion del texto del parrafo que va en cursiva."""
    total = len(p.get_text(strip=True))
    if total == 0:
        return 0.0
    ital = sum(len(t.get_text(strip=True)) for t in p.find_all(["em", "i"]))
    return ital / total
 
 
def leading_bold_label(p: Tag) -> str | None:
    """
    Devuelve la etiqueta de hablante si el parrafo EMPIEZA con negrita
    seguida (dentro o fuera) de dos puntos. Ignora negritas interiores.
    """
    first = None
    for child in p.children:
        if isinstance(child, Tag):
            first = child
            break
        if isinstance(child, str) and child.strip():
            return None          # empieza con texto plano -> continuacion
    if first is None or first.name not in ("strong", "b"):
        return None
 
    label = first.get_text(strip=True).rstrip(":").strip()
    # los dos puntos pueden ir fuera de la negrita: "**Draghi**: texto"
    tail = (first.next_sibling or "")
    tail = tail if isinstance(tail, str) else ""
    if not (first.get_text(strip=True).endswith(":") or tail.lstrip().startswith(":")):
        return None
    if len(label) > 40 or not label:
        return None
    return label
 
 
def strip_label(p: Tag, label: str) -> str:
    txt = p.get_text(" ", strip=True)
    return re.sub(rf"^{re.escape(label)}\s*:\s*", "", txt).strip()
 
 
def find_qa_start(soup: BeautifulSoup, paras: list) -> int:
    """
    Indice del primer parrafo del Q&A. -1 si no se encuentra.
 
    Prioridad:
      1. ancla id="qa"  (el BCE enlaza "Jump to the transcript of the Q&A")
      2. frase de cierre de la declaracion introductoria
      3. primer parrafo con etiqueta de hablante del BCE
    """
    anchor = soup.find(id="qa")
    if anchor is not None:
        after = anchor.find_all_next("p")
        if after:
            for i, p in enumerate(paras):
                if p is after[0]:
                    return i
 
    pat = re.compile("|".join(QA_ANCHORS), re.I)
    for i, p in enumerate(paras):
        if pat.search(p.get_text(" ", strip=True)):
            return i + 1
 
    for i, p in enumerate(paras):
        lab = leading_bold_label(p)
        if lab and lab.lower() in ECB_POLICYMAKERS:
            return i
    return -1

In [7]:
# ---------------------------------------------------------------- parseo
def parse_qa(paras: list) -> list[dict]:
    """
    Recorre los parrafos del Q&A acumulando:
      - bloques de pregunta  (cursiva, posiblemente varios parrafos)
      - bloques de respuesta (etiqueta en negrita + parrafos planos siguientes)
    """
    pairs, q_block, cur = [], [], None
    fresh = False   # True mientras se acumulan parrafos de una misma pregunta
 
    for p in paras:
        # CURSIVA PRIMERO: las preguntas son <em><strong>...</strong></em>, o sea
        # negrita Y cursiva a la vez. Si se comprueba la negrita antes, se
        # clasifican como respuestas.
        is_ital = italic_ratio(p) > 0.6
        label = None if is_ital else leading_bold_label(p)
 
        if is_ital:
            if cur:
                pairs.append(cur)
                cur = None
            if not fresh:                 # arranca bloque de pregunta nuevo
                q_block, fresh = [], True
            q = re.sub(r"^Question\s*:\s*", "", p.get_text(" ", strip=True), flags=re.I)
            q_block.append(q)
 
        elif label and label.lower() in ECB_POLICYMAKERS:
            if cur:
                pairs.append(cur)
            fresh = False                 # la pregunta queda fijada y PERSISTE:
            cur = {                       # si Draghi y el vicepresidente contestan
                "speaker": label,         # a la misma pregunta, ambas filas la heredan
                "question": " ".join(q_block).strip(),
                "answer": [strip_label(p, label)],
            }
 
        elif label and label.lower() in NON_POLICYMAKERS:
            if cur:                       # moderador -> cierra la respuesta abierta
                pairs.append(cur)
                cur = None
 
        else:
            if cur:                       # parrafo plano -> continua la respuesta
                cur["answer"].append(p.get_text(" ", strip=True))
            # si no hay respuesta abierta, es ruido; se descarta
 
    if cur:
        pairs.append(cur)
 
    for pr in pairs:
        pr["answer"] = " ".join(pr["answer"]).strip()
    return [pr for pr in pairs if pr["answer"]]
 
 
def parse_one(date: str) -> dict:
    soup = get_soup(date)
    paras = article_paragraphs(soup)
    qa_i = find_qa_start(soup, paras)
 
    if qa_i == -1:
        print(f"  AVISO {date}: no se localiza el inicio del Q&A")
        remarks_paras, qa = paras, []
    else:
        remarks_paras = paras[:qa_i]
        qa = parse_qa(paras[qa_i:], date)
 
    texts = [p.get_text(" ", strip=True) for p in remarks_paras]
    texts = [t for t in texts if len(t) >= MIN_PARA_CHARS]
 
    return {"date": date, "remarks_paras": texts, "qa": qa}

In [8]:
# ---------------------------------------------------------------- salida
def era_of(date: str) -> str:
    return "draghi" if date < "2019-11-01" else "lagarde"
 
 
def build_excel(data):
    rows_rem, rows_rem_p, rows_ans, rows_ans_p = [], [], [], []   # <-- rows_ans_p

    for d in data:
        date, paras = d["date"], d["remarks_paras"]
        rows_rem.append({"date": date, "era": era_of(date), "text": " ".join(paras)})
        for k, t in enumerate(paras, 1):
            rows_rem_p.append({"date": date, "order": k, "text": t})

        for k, qa in enumerate(d["qa"], 1):
            item = f"ECB_{date.replace('-','')}_A{k:02d}"
            rows_ans.append({"date": date, "item": item, "speaker": qa["speaker"],
                             "src": qa["src"], "question": qa["question"],
                             "text": qa["answer"]})
            for j, para in enumerate(qa["answer_paras"], 1):       # <-- bucle nuevo
                rows_ans_p.append({"date": date, "item": item, "order": j,
                                   "speaker": qa["speaker"], "text": para})

    with pd.ExcelWriter(OUTPUT, engine="openpyxl") as w:
        pd.DataFrame(rows_rem).to_excel(w, sheet_name="remarks", index=False)
        pd.DataFrame(rows_rem_p).to_excel(w, sheet_name="remarks_para", index=False)
        pd.DataFrame(rows_ans).to_excel(w, sheet_name="answers", index=False)
        pd.DataFrame(rows_ans_p).to_excel(w, sheet_name="answers_para", index=False)  # <-- hoja nueva

    print(f"\nEscrito {OUTPUT}")
    print(f"  answers: {len(rows_ans)} | answers_para: {len(rows_ans_p)}")
    return pd.DataFrame(rows_ans)

In [9]:
# ---------------------------------------------------------------- validacion
def validate(ans: pd.DataFrame) -> None:
    ans["nw"] = ans.text.str.split().str.len()
    ans["year"] = pd.to_datetime(ans.date).dt.year
 
    print("\n--- respuestas por conferencia y palabras por respuesta ---")
    g = ans.groupby("year").agg(
        confs=("date", "nunique"), rows=("text", "size"), med_words=("nw", "median")
    )
    g["per_conf"] = (g["rows"] / g["confs"]).round(1)
    print(g.to_string())
 
    print("\nReferencia Gorodnichenko (FOMC): 19,2 respuestas/conf, mediana 219 palabras")
 
    per_conf = ans.groupby("date").size()
    bad = per_conf[per_conf < 8]
    if len(bad):
        print(f"\nCONFERENCIAS SOSPECHOSAS (<8 respuestas): {len(bad)}")
        print(bad.to_string())
    else:
        print("\nNinguna conferencia por debajo de 8 respuestas.")
 
    huge = ans[ans.nw > 900]
    if len(huge):
        print(f"\nRespuestas >900 palabras (posible fusion): {len(huge)}")
        print(huge[["date", "item", "nw"]].head(15).to_string(index=False))
 
    sin_q = (ans.question.str.len() == 0).sum()
    print(f"\nRespuestas sin pregunta emparejada: {sin_q} / {len(ans)}")
 

In [10]:
# ---------------------------------------------------------------- diagnostico
def diagnose(dates: list[str]) -> None:
    for date in dates:
        print("=" * 70)
        print(date)
        soup = get_soup(date)
        paras = article_paragraphs(soup)
        qa_i = find_qa_start(soup, paras)
        print(f"  parrafos totales={len(paras)}  inicio Q&A={qa_i}")
        if qa_i == -1:
            continue
        for p in paras[qa_i:qa_i + 10]:
            ir = italic_ratio(p)
            lab = None if ir > 0.6 else leading_bold_label(p)
            kind = "QUES" if ir > 0.6 else ("ANS " if lab else "cont")
            print(f"  [{kind}] lab={lab!r} ital={ir:.2f} | {p.get_text(' ', strip=True)[:85]}")
 
 
def list_speakers(dates: list[str]) -> None:
    from collections import Counter
    c = Counter()
    for date in dates:
        for p in article_paragraphs(get_soup(date)):
            lab = leading_bold_label(p)
            if lab:
                c[lab] += 1
    print("Etiquetas de hablante encontradas en el corpus:\n")
    for lab, n in c.most_common():
        known = lab.lower().strip(":") in ECB_POLICYMAKERS
        print(f"  {n:5d}  {lab!r}  {'' if known else '  <-- NO en ECB_POLICYMAKERS'}")

In [11]:
dates = pd.read_csv(URL_MASTER)["target_date"].astype(str).tolist()
faltan = [d for d in dates if not local_path(d).exists()]
print(f"{len(dates)-len(faltan)}/{len(dates)} encontrados")
print("faltan:", faltan)

104/104 encontrados
faltan: []


In [12]:
def dump_raw(date, n=8):
    soup = get_soup(date)
    paras = article_paragraphs(soup)
    i = find_qa_start(soup, paras)
    print("=" * 70); print(date)
    for k, p in enumerate(paras[i:i+n]):
        total = len(p.get_text(strip=True))
        bold = sum(len(t.get_text(strip=True)) for t in p.find_all(["strong", "b"]))
        ital = sum(len(t.get_text(strip=True)) for t in p.find_all(["em", "i"]))
        br = bold/total if total else 0
        ir = ital/total if total else 0
        print(f"\n--- p[{k}]  bold={br:.2f}  ital={ir:.2f}")
        print("   ", p.decode()[:260].replace("\n", " "))

dump_raw("2018-01-25")
dump_raw("2024-01-25")

2018-01-25

--- p[0]  bold=0.00  ital=0.00
    <p>* * *</p>

--- p[1]  bold=1.00  ital=0.00
    <p><strong>I have a question concerning the asset purchase programme. When you launched the programme you decided to keep the purchases as strictly according to the ECB capital key. But now it turns out that you are increasingly deviating from this as a new su

--- p[2]  bold=0.00  ital=0.00
    <p>Draghi: It's not so, it doesn't affect the neutrality of the purchases – and I'll tell you the reasons why you observe these deviations. The ECB doesn't favour certain countries over others in its PSPP purchase programme implementation. As you know, purchas

--- p[3]  bold=0.00  ital=0.00
    <p>Of course the largest deviation from the ECB's capital key is Greece due to their ineligibility. In addition Eurosystem holdings of Portuguese securities are 1.8%, which is 27% below, compared to [the Portuguese] share of ECB capital key, which is 2.48%. Th

--- p[4]  bold=0.00  ital=0.00
    <p>So, all th

In [13]:
SEPARATOR = re.compile(r"^[\s*\-–—_.·•]+$")
LABEL     = re.compile(r"^\s*([A-Za-zÀ-ÿ'’.\- ]{2,25}?)\s*:\s+")

def bold_ratio(p):
    """Fraccion del parrafo en negrita o cursiva. Las preguntas del BCE van
    integramente marcadas; las respuestas van en texto plano. Vale para 2013
    (<em><strong>), 2018 (<strong>) y 2024 (<strong>)."""
    total = len(p.get_text(strip=True))
    if total == 0:
        return 0.0
    return sum(len(t.get_text(strip=True))
               for t in p.find_all(["strong", "b", "em", "i"])) / total

def is_separator(p):
    return bool(SEPARATOR.match(p.get_text(strip=True)))   # filtra "* * *"

def president_on(date):
    return "Draghi" if date < "2019-11-01" else "Lagarde"

def parse_qa(paras, date):
    pairs, q_block, cur = [], [], None
    fresh = False
    for p in paras:
        if is_separator(p):
            continue
        txt = p.get_text(" ", strip=True)

        if bold_ratio(p) > 0.6:                       # PREGUNTA
            if cur:
                pairs.append(cur); cur = None
            if not fresh:
                q_block, fresh = [], True
            q_block.append(re.sub(r"^Question\s*:\s*", "", txt, flags=re.I))
            continue

        m = LABEL.match(txt)
        if m and m.group(1).lower().strip() in ECB_POLICYMAKERS:   # respuesta con prefijo
            if cur:
                pairs.append(cur)
            fresh = False
            cur = {"speaker": m.group(1).strip(), "src": "prefijo",
                   "question": " ".join(q_block).strip(),
                   "answer": [txt[m.end():].strip()]}
        elif cur is None and q_block:                 # respuesta sin prefijo (era Lagarde)
            fresh = False
            cur = {"speaker": president_on(date), "src": "imputado",
                   "question": " ".join(q_block).strip(), "answer": [txt]}
        elif cur:                                     # continuacion de la respuesta
            cur["answer"].append(txt)

    if cur:
        pairs.append(cur)
    for pr in pairs:
        pr["answer"] = " ".join(pr["answer"]).strip()
    return [pr for pr in pairs if pr["answer"]]

In [14]:
soup = get_soup("2020-03-12")
paras = article_paragraphs(soup)
print("parrafos:", len(paras))
for i, p in enumerate(paras):
    b = bold_ratio(p)
    if b > 0.6 or i > len(paras) - 3:
        print(f"[{i}] bold={b:.2f} | {p.get_text(' ', strip=True)[:80]}")

parrafos: 75
[22] bold=1.00 | My first question would be about the rationale for not cutting interest rates to
[23] bold=1.00 | Another question is about the issue you mentioned as the last, which is this ext
[26] bold=1.00 | A question on the economy with the risks pointed clearly to the downside. Can th
[27] bold=1.00 | What will happen with the strategy review; will it be postponed or put on hold, 
[30] bold=1.00 | I'd like to hear your response to the travel restrictions announced by President
[31] bold=1.00 | Secondly, just to go back to a point that was already raised on issuer limits, d
[34] bold=1.00 | First, is this your do-whatever-it-takes moment, and did you expect to have one 
[35] bold=1.00 | The second point is, by not cutting rates today it makes the ECB look a little o
[38] bold=1.00 | The additional net asset purchase's limits, such as the issuer limits, might bec
[39] bold=1.00 | Does the decision not to reduce the deposit rate also have to do with the change
[42] bo

In [15]:
# ---------------------------------------------------------------- hablantes
SURNAMES = {"draghi","lagarde","trichet","constancio","constâncio","guindos","praet",
            "coeure","cœuré","coeuré","mersch","lautenschlager","lautenschläger",
            "lane","schnabel","elderson","panetta","cipollone","buch",
            # gobernadores nacionales (miembros del Consejo de Gobierno)
            "visco","georghadji","bonnici","nowotny","stournaras","noyer","coene"}
EXCLUDE = {"graeff","question","questions","moderator"}
TITLES  = re.compile(r"^(?:mr|ms|mrs|dr|president|vice[-\s]president)\.?\s+", re.I)

def match_speaker(lab):
    """Normaliza titulos y nombres de pila; devuelve la etiqueta si es del BCE."""
    s = TITLES.sub("", lab.strip()).strip().lower()
    if not s or s in EXCLUDE:
        return None
    if s in SURNAMES:
        return lab.strip()
    toks = s.replace("-", " ").split()
    if toks and toks[-1] in SURNAMES and len(toks) <= 3:   # "Luis de Guindos"
        return lab.strip()
    return None

# ---------------------------------------------------------------- basura de pagina
BOILERPLATE = re.compile(
    r"^(disclaimer|reproduction is permitted|please note that related topic|"
    r"copyright|media contacts?)\b", re.I)

def is_boilerplate(p):
    return bool(BOILERPLATE.match(p.get_text(" ", strip=True)))

# ---------------------------------------------------------------- localizar Q&A
def find_qa_start(soup, paras):
    anchor = soup.find(id="qa")
    if anchor is not None:
        after = anchor.find_all_next("p")
        if after:
            for i, p in enumerate(paras):
                if p is after[0]:
                    return i
    pat = re.compile("|".join(QA_ANCHORS), re.I)
    for i, p in enumerate(paras):
        if pat.search(p.get_text(" ", strip=True)):
            return i + 1
    # respaldo: primer parrafo en negrita (= primera pregunta). Cubre 2020-03-12.
    for i, p in enumerate(paras):
        if i >= 5 and bold_ratio(p) > 0.6:
            return i
    return -1

# ---------------------------------------------------------------- Q&A
def parse_qa(paras, date):
    pairs, q_block, cur = [], [], None
    fresh = False
    for p in paras:
        if is_boilerplate(p):
            break                                   # pie de pagina: se acabo el Q&A
        if is_separator(p):
            continue
        txt = p.get_text(" ", strip=True)

        if bold_ratio(p) > 0.6:                     # PREGUNTA
            if cur:
                pairs.append(cur); cur = None
            if not fresh:
                q_block, fresh = [], True
            q_block.append(re.sub(r"^Question\s*:\s*", "", txt, flags=re.I))
            continue

        m = LABEL.match(txt)
        sp = match_speaker(m.group(1)) if m else None

        if sp:                                      # respuesta con prefijo
            if cur:
                pairs.append(cur)
            fresh = False
            cur = {"speaker": sp, "src": "prefijo",
                   "question": " ".join(q_block).strip(),
                   "answer": [txt[m.end():].strip()]}
        elif cur is None and q_block:               # respuesta sin prefijo (era Lagarde)
            fresh = False
            cur = {"speaker": president_on(date), "src": "imputado",
                   "question": " ".join(q_block).strip(), "answer": [txt]}
        elif cur:
            cur["answer"].append(txt)

    if cur:
        pairs.append(cur)
    for pr in pairs:
        pr["answer_paras"] = list(pr["answer"])          # <-- AÑADE esta linea
        pr["answer"] = " ".join(pr["answer"]).strip()
    return [pr for pr in pairs if pr["answer"]]

In [16]:
def parse_one(date):
    soup = get_soup(date)
    paras = [p for p in article_paragraphs(soup) if not is_boilerplate(p)]
    qa_i = find_qa_start(soup, paras)
    if qa_i == -1:
        raise RuntimeError(f"{date}: no se localiza el inicio del Q&A")
    texts = [p.get_text(" ", strip=True) for p in paras[:qa_i]]
    texts = [t for t in texts if len(t) >= MIN_PARA_CHARS]
    return {"date": date, "remarks_paras": texts, "qa": parse_qa(paras[qa_i:], date)}

In [17]:
data = [parse_one(d) for d in dates if local_path(d).exists()]
print(f"Conferencias parseadas: {len(data)}")
ans = build_excel(data)
validate(ans)

Conferencias parseadas: 104

Escrito C:\Users\PC\Desktop\TFM\BIG DATA\TEXT SENTIMENT\data\ECB_TEXT (2013-2024)_v2.xlsx
  answers: 1543 | answers_para: 3223

--- respuestas por conferencia y palabras por respuesta ---
      confs  rows  med_words  per_conf
year                                  
2013     12   238      161.5      19.8
2014     12   175      249.0      14.6
2015      8   142      165.5      17.8
2016      8   131      182.0      16.4
2017      8   145      129.0      18.1
2018      8   123      177.0      15.4
2019      8   142      201.5      17.8
2020      8    85      350.0      10.6
2021      8    86      337.0      10.8
2022      8    98      320.0      12.2
2023      8    98      270.0      12.2
2024      8    80      304.0      10.0

Referencia Gorodnichenko (FOMC): 19,2 respuestas/conf, mediana 219 palabras

CONFERENCIAS SOSPECHOSAS (<8 respuestas): 1
date
2023-12-14    6

Respuestas >900 palabras (posible fusion): 14
      date             item   nw
2013-05-02 ECB

In [18]:
import numpy as np

ans["nw"]  = ans.text.str.split().str.len()
ans["era"] = np.where(ans.date < "2019-11-01", "draghi", "lagarde")

por_conf = ans.groupby(["era", "date"]).nw.sum()
print("PALABRAS TOTALES DE Q&A POR CONFERENCIA")
print(por_conf.groupby("era").describe()[["50%", "mean", "min", "max"]].round(0))

PALABRAS TOTALES DE Q&A POR CONFERENCIA
            50%    mean     min     max
era                                    
draghi   3716.0  3651.0  2293.0  4708.0
lagarde  3848.0  3779.0  2367.0  4723.0


In [18]:
diagnose(["2023-12-14"])

2023-12-14
  parrafos totales=46  inicio Q&A=26
  [cont] lab=None ital=0.00 | I have two questions. The first one is on the market expectations about the future ra
  [cont] lab=None ital=0.00 | Thank you very much for your two questions. But let me remind you of one thing. We ar
  [cont] lab=None ital=0.00 | Should we lower our guard? We asked ourselves that question. No, we should absolutely
  [cont] lab=None ital=0.00 | When we look at the data that we have now, it is not declining. Compensation per empl
  [cont] lab=None ital=0.00 | Last night, J. Powell said that rate cuts have started to come into view and and were
  [cont] lab=None ital=0.00 | Thank you very much for your two questions. On the first one, it is very easy. We did
  [cont] lab=None ital=0.00 | I have to come back to the rate outlook. You mentioned that a lot of data will arrive
  [cont] lab=None ital=0.00 | On your latter question, we don't have any particular country in mind. I think we hav
  [cont] lab=None ital=0

In [19]:
tot = ans.groupby("date").nw.sum()
print("2023-12-14:", tot.loc["2023-12-14"], "palabras")
print("mediana de todas:", tot.median())
print("\nlas 5 conferencias con menos texto:")
print(tot.nsmallest(5))

2023-12-14: 2367 palabras
mediana de todas: 3784.0

las 5 conferencias con menos texto:
date
2018-07-26    2293
2016-10-20    2338
2023-12-14    2367
2017-01-19    2369
2024-12-12    2781
Name: nw, dtype: int64


In [20]:
def diagnose(fechas):
    for date in fechas:
        soup = get_soup(date)
        paras = [p for p in article_paragraphs(soup) if not is_boilerplate(p)]
        i = find_qa_start(soup, paras)
        print("=" * 70); print(f"{date} | parrafos={len(paras)} | inicio Q&A={i}")
        if i == -1:
            continue
        for p in paras[i:i+10]:
            if is_separator(p):
                continue
            b = bold_ratio(p)
            m = LABEL.match(p.get_text(" ", strip=True))
            sp = match_speaker(m.group(1)) if m else None
            kind = "QUES" if b > 0.6 else ("ANS " if sp else "cont")
            print(f"  [{kind}] bold={b:.2f} sp={sp} | {p.get_text(' ', strip=True)[:70]}")

In [21]:
tot = ans.groupby("date").nw.sum()
bajas = tot[tot < 1800]     # ~la mitad de la mediana

In [22]:
ap = pd.read_excel(OUTPUT, sheet_name="answers_para", engine="openpyxl")
ap["nw"] = ap.text.str.split().str.len()
print(ap.nw.describe().round(0))
print("parrafos >380 palabras (~512 tokens):", (ap.nw > 380).sum())

count    3223.0
mean      119.0
std        84.0
min         1.0
25%        64.0
50%       106.0
75%       156.0
max      1200.0
Name: nw, dtype: float64
parrafos >380 palabras (~512 tokens): 47


In [23]:
cortos = (ap.text.str.split().str.len() < 5).sum()
print(cortos, "de", len(ap))

28 de 3223


In [24]:
cortos = ap[ap.text.str.split().str.len() < 5]
print(cortos[["date", "item", "text"]].to_string(index=False))

      date             item                         text
2013-06-06 ECB_20130606_A13            Well, not really.
2013-11-07 ECB_20131107_A02                    May 2003.
2013-11-07 ECB_20131107_A05              No, I am sorry.
2013-11-07 ECB_20131107_A08                 Yes, indeed.
2014-11-06 ECB_20141106_A14                         Yes.
2016-03-10 ECB_20160310_A08            It's not indexed.
2016-12-08 ECB_20161208_A18                     Exactly.
2017-04-27 ECB_20170427_A12                          No.
2017-06-08 ECB_20170608_A09     Single Resolution Board?
2017-06-08 ECB_20170608_A15          It's purely formal.
2017-09-07 ECB_20170907_A03                     I don't.
2017-09-07 ECB_20170907_A07                          No.
2017-10-26 ECB_20171026_A12       On the first question.
2017-10-26 ECB_20171026_A14                          No.
2017-10-26 ECB_20171026_A15                    No? Okay.
2017-10-26 ECB_20171026_A18      Second one, me neither.
2017-10-26 ECB_20171026_A19    

In [25]:
print(ans[ans.item == "ECB_20190124_A04"][["speaker","src","question","text"]].to_string())

    speaker      src                           question        text
957  Draghi  prefijo  The answer to my second question.  Which was?
